## Step 1
Merge volume bids with generator info  
Input: s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/  
Output: s3://thesis--ec331-s3/enriched-volume-bids/

In [2]:
import awswrangler as wr
import pandas as pd
import time
import os
import gc
from datetime import datetime

# Path to generator info file
generator_info_path = "s3://thesis--ec331-s3/AEMO-Participants - Sheet1.csv"

# Read generator info
try:
    generator_info_df = wr.s3.read_csv(path=generator_info_path)
    print(f"Successfully read generator info: {generator_info_path}")
    print(f"Generator info shape: {generator_info_df.shape}")
    print("Generator info columns:", generator_info_df.columns.tolist())
except Exception as e:
    print(f"Error reading generator info: {e}")
    generator_info_df = None

# Adjusted input path from which to merge
s3_input_path = (
    "s3://thesis--ec331-s3/step-2-capped-volume-bids/"
    "RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/"
    "3006843706904e8fa6101fd5b46ee327.snappy.parquet"
)

# Read the main dataframe from the provided S3 path
try:
    df = wr.s3.read_parquet(path=s3_input_path)
    print(f"Successfully read input data from {s3_input_path}")
    print(f"Input data shape: {df.shape}")
except Exception as e:
    print(f"Error reading input data: {e}")
    df = pd.DataFrame()  # or handle the error as needed

# Extract the folder name from the input path so it can be preserved in the output path.
# This assumes the input path structure is:
# s3://thesis--ec331-s3/step-2-capped-volume-bids/<folder>/<filename>
input_prefix = "s3://thesis--ec331-s3/step-2-capped-volume-bids/"
relative_path = s3_input_path[len(input_prefix):]  # e.g. "RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/3006843706904e8fa6101fd5b46ee327.snappy.parquet"
folder_name = os.path.dirname(relative_path)       # e.g. "RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped"

# Construct the base output folder that preserves the original folder name
output_base_folder = f"s3://thesis--ec331-s3/enriched-volume-bids/{folder_name}"

# Define chunk processing function
def process_chunk(chunk_df, generator_info_df, chunk_num, total_chunks, timestamp):
    try:
        print(f"Processing chunk {chunk_num}/{total_chunks} with {len(chunk_df)} rows")
        
        # Merge the chunk with generator info using 'DUID' as key
        if 'DUID' in chunk_df.columns and generator_info_df is not None and 'DUID' in generator_info_df.columns:
            enriched_chunk = pd.merge(
                chunk_df,
                generator_info_df,
                on='DUID',
                how='left'
            )
            
            # Construct output filename and path preserving the folder name
            chunk_output_filename = f"enriched_volume_bids_{timestamp}_chunk{chunk_num}of{total_chunks}.parquet"
            chunk_output_path = f"{output_base_folder}/{chunk_output_filename}"
            
            # Save this enriched chunk to S3
            chunk_start_time = time.time()
            wr.s3.to_parquet(
                df=enriched_chunk,
                path=chunk_output_path,
                index=False,
                compression="snappy"
            )
            
            chunk_elapsed_time = time.time() - chunk_start_time
            print(f"✓ Chunk {chunk_num}/{total_chunks} saved to {chunk_output_path}")
            print(f"  Chunk save completed in {chunk_elapsed_time:.2f} seconds")
            print(f"  Chunk size: {len(enriched_chunk)} rows, {enriched_chunk.shape[1]} columns")
            
            # Clear memory
            del enriched_chunk
            
        else:
            print("Skipping chunk - missing required columns")
            
    except Exception as e:
        print(f"Error processing chunk {chunk_num}: {str(e)}")
    
    # Force garbage collection
    gc.collect()

# Define a small chunk size to avoid memory issues
chunk_size = 10000

# Calculate total rows and chunks from the main dataframe
total_rows = len(df)
total_chunks = (total_rows + chunk_size - 1) // chunk_size

# Create a timestamp string for output filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Processing {total_rows} rows in {total_chunks} chunks of {chunk_size} rows each")

# Process the data in chunks
for i in range(total_chunks):
    start_idx = i * chunk_size
    end_idx = min(start_idx + chunk_size, total_rows)
    
    # Extract the chunk (copy to avoid referencing the original DataFrame)
    chunk = df.iloc[start_idx:end_idx].copy()
    
    # Process this chunk by merging with generator info and writing to S3
    process_chunk(chunk, generator_info_df, i+1, total_chunks, timestamp)
    
    # Clean up memory
    del chunk
    gc.collect()
    
    # Optionally, print memory usage stats if psutil is installed
    try:
        import psutil
        process = psutil.Process(os.getpid())
        print(f"Memory usage after chunk {i+1}: {process.memory_info().rss / 1e9:.2f} GB")
    except:
        pass

print("All chunks processed successfully")

Successfully read generator info: s3://thesis--ec331-s3/AEMO-Participants - Sheet1.csv
Generator info shape: (532, 20)
Generator info columns: ['Participant', 'Station Name', 'Region', 'Dispatch Type', 'Category', 'Classification', 'Fuel Source - Primary', 'Fuel Source - Descriptor', 'Technology Type - Primary', 'Technology Type - Descriptor', 'Units', 'Aggregation', 'DUID', 'Reg Cap generation (MW)', 'Max Cap generation (MW)', 'Max ROC/Min generation', 'Reg Cap consumption (MW)', 'Max Cap consumption (MW)', 'Max ROC/Min consumption', 'Comments']
Successfully read input data from s3://thesis--ec331-s3/step-2-capped-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/3006843706904e8fa6101fd5b46ee327.snappy.parquet
Input data shape: (41760, 28)
Processing 41760 rows in 5 chunks of 10000 rows each
Processing chunk 1/5 with 10000 rows
✓ Chunk 1/5 saved to s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/enriched_volume_bids_2025

In [18]:
import pandas as pd

# Show all columns without truncation
pd.set_option('display.max_columns', None)

# Display the first 5 rows again
print(enriched_df.tail())

          I  BIDS  BIDOFFERPERIOD    1      DUID    BIDTYPE  \
21021115  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021116  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021117  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021118  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   
21021119  D  BIDS  BIDOFFERPERIOD  1.0  WALGRVG1  RAISE1SEC   

                  TRADINGDATE        OFFERDATETIME  PERIODID  MAXAVAIL  \
21021115  2023/11/15 00:00:00  2023/11/16 03:51:42     284.0      27.0   
21021116  2023/11/15 00:00:00  2023/11/16 03:51:42     285.0      27.0   
21021117  2023/11/15 00:00:00  2023/11/16 03:51:42     286.0      27.0   
21021118  2023/11/15 00:00:00  2023/11/16 03:51:42     287.0      27.0   
21021119  2023/11/15 00:00:00  2023/11/16 03:51:42     288.0      27.0   

          FIXEDLOAD  RAMPUPRATE  RAMPDOWNRATE  ENABLEMENTMIN  ENABLEMENTMAX  \
21021115        NaN         NaN           NaN            0.0           50.0   
21021116        Na

In [20]:
# Filtering rows where 'Region Dispatch' is NOT NaN
filtered_df = enriched_df[enriched_df['Region'].notna()]

# Display the last 5 rows of the filtered dataframe
print(filtered_df.tail())

          I  BIDS  BIDOFFERPERIOD    1     DUID    BIDTYPE  \
18440923  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440924  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440925  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440926  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   
18440927  D  BIDS  BIDOFFERPERIOD  1.0  QBYNBG1  RAISE1SEC   

                  TRADINGDATE        OFFERDATETIME  PERIODID  MAXAVAIL  \
18440923  2023/11/15 00:00:00  2023/11/16 03:48:32     285.0       5.0   
18440924  2023/11/15 00:00:00  2023/11/16 03:48:32     286.0       5.0   
18440925  2023/11/15 00:00:00  2023/11/16 03:48:32     287.0       5.0   
18440926  2023/11/15 00:00:00  2023/11/16 03:48:32     288.0       5.0   
18440927  2023/11/15 00:00:00  2023/11/16 02:08:31     288.0       5.0   

          FIXEDLOAD  RAMPUPRATE  RAMPDOWNRATE  ENABLEMENTMIN  ENABLEMENTMAX  \
18440923        NaN         NaN           NaN            0.0           10.0   
18440924        NaN     

In [21]:
print(enriched_df.isna().sum())

I                                      0
BIDS                                   0
BIDOFFERPERIOD                         0
1                                      0
DUID                                   0
BIDTYPE                                0
TRADINGDATE                            0
OFFERDATETIME                          0
PERIODID                               0
MAXAVAIL                               0
FIXEDLOAD                       21021120
RAMPUPRATE                      21021120
RAMPDOWNRATE                    21021120
ENABLEMENTMIN                          0
ENABLEMENTMAX                          0
LOWBREAKPOINT                          0
HIGHBREAKPOINT                         0
BANDAVAIL1                             0
BANDAVAIL2                             0
BANDAVAIL3                             0
BANDAVAIL4                             0
BANDAVAIL5                             0
BANDAVAIL6                             0
BANDAVAIL7                             0
BANDAVAIL8      